## Load the required packages

In [1]:
import anndata
from scipy.io import mmwrite
import pandas as pd
import os
from scipy.sparse import csr_matrix
from pathlib import Path
import numpy as np

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/scMOG"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")   
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

## Create the saved format required to run scMOG

In [6]:
gene_expression_train = csr_matrix(train_data.X.T)
gene_expression_test = csr_matrix(test_data.X.T)

protein_expression_train = csr_matrix(train_protein_expression.T.values)
protein_expression_test = csr_matrix(test_protein_expression.T.values)

genes = pd.Series(list(train_data.var.index))
proteins = pd.Series(list(train_data.uns["protein_names"]))

train_cells = pd.Series(list(train_data.obs.index))
test_cells = pd.Series(list(test_data.obs.index))

In [7]:
subdirs = ["train_rna", "test_rna", "train_protein", "test_protein"]
for sub in subdirs:
    os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

In [8]:
mmwrite(f"{base_dir}/train_rna/count.mtx", gene_expression_train)
mmwrite(f"{base_dir}/test_rna/count.mtx", gene_expression_test)
mmwrite(f"{base_dir}/train_protein/count.mtx", protein_expression_train)
mmwrite(f"{base_dir}/test_protein/count.mtx", protein_expression_test)

genes.to_csv(f"{base_dir}/train_rna/gene.csv", index=False, header=False)
genes.to_csv(f"{base_dir}/test_rna/gene.csv", index=False, header=False)
proteins.to_csv(f"{base_dir}/train_protein/protein.csv", index=False, header=False)
proteins.to_csv(f"{base_dir}/test_protein/protein.csv", index=False, header=False)

train_cells.to_csv(f"{base_dir}/train_rna/barcode.csv", index=False, header=False)
train_cells.to_csv(f"{base_dir}/train_protein/barcode.csv", index=False, header=False)
test_cells.to_csv(f"{base_dir}/test_rna/barcode.csv", index=False, header=False)
test_cells.to_csv(f"{base_dir}/test_protein/barcode.csv", index=False, header=False)